In [1]:
from pathlib import Path
import sys
import os
import clingo
project_root = "."


In [2]:
from typing import Any

INSTANCE = "population"

ASPGARP_FILES = [
    f"{project_root}/prototype/single/qsim.lp",
    f"{project_root}/prototype/single/cd.lp",
    f"{project_root}/prototype/adapter.lp",
    f"{project_root}/prototype/db.lp",
    f"{project_root}/prototype/preds.lp",
    f"{project_root}/prototype/single/garp_dynamics.lp",
    f"{project_root}/prototype/qcn/encoding.lp",
    f"{project_root}/prototype/qcn/calculi/point.lp",

]

REFERENCE_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp",
    f"{project_root}/models/{INSTANCE}/ref.lp"
]

BASIC_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp"
]

META_FILES = [
    f"{project_root}/prototype/meta.lp",
]

OUTPUT = f"{project_root}/out/{INSTANCE}"

MAX_STATES = "0"

#create output directory if it doesn't exist
os.makedirs(OUTPUT, exist_ok=True)

In [3]:
MODEL_DB = REFERENCE_MODEL[0]
MODEL_REF = REFERENCE_MODEL[1]
!clingo 1 --project --warn=none --outf=2 {MODEL_DB} {MODEL_REF} | clingraph --out=render --format=svg --viz-encoding={project_root}/prototype/viz/causal.lp --dir={OUTPUT}/visuals --default-graph=causal

./prototype/viz/causal.lp:28:18-27: info: atom does not occur in any rule head:
  qvar(V,#Anon0)

./prototype/viz/causal.lp:29:34-43: info: atom does not occur in any rule head:
  qvar(V,#Anon0)

-> Image for graph causal, saved in: ./out/population/visuals/0/causal.svg


In [4]:
import clingo


def states_from_reference_model(
    RMODEL: list[str], ASPGARPFILES: list[str]
) -> list[dict[str, tuple[str, str]]]:
    states: list[dict[str, tuple[str, str]]] = []

    ctl = clingo.Control([f"{MAX_STATES}", "--project", "--warn=none"])
    for path in RMODEL + ASPGARPFILES:
        ctl.load(path)

    ctl.add("base", [], "#show obs/1.")
    ctl.add("base", [], ":- violation.")
    ctl.ground([("base", [])])

    with ctl.solve(yield_=True) as handle:
        for model in handle:
            state: dict[str, tuple[str, str]] = {}
            for atom in model.symbols(shown=True):
                if atom.name == "obs" and len(atom.arguments) == 1:
                    if len(atom.arguments[0].arguments) == 3:
                        varname, value, direction = atom.arguments[0].arguments
                        state[str(varname)] = (value, str(direction))
                    elif len(atom.arguments[0].arguments) == 2:
                        varname, value = atom.arguments[0].arguments
                        state[str(varname)] = [value]
            states.append(state)

    return states


def states_to_description(
    states: list[dict[str, tuple[str, str]]]
) -> list[str]:
    descriptions: list[str] = []

    for i, state in enumerate(states):
        #state is either a pair ot a triple, depending on whether it's a holds or holds_relation
        if list(state.values()) == 3:
            parts = [f"obsT(holds(SID,{varname},{value},{direction}))"
                 for varname, (value, direction) in state.items()]
        elif list(state.values()) == 2:
            parts = [f"obs(holds_relation(SID,{varname},{value})))"
                 for varname, value in state.items()]
        description = f"state(SID,{i}) :- " + ", ".join(parts) + "."
        descriptions.append(description)

    return descriptions

def states_to_labels(states: list[dict[str, tuple[str, str]]]) -> list[str]:
    labels: list[str] = []

    for i, state in enumerate(states):
        parts = [f"{varname}=({value},{direction})"
                 for varname, (value, direction) in state.items()]
        label = f"statelabel({i},\"" + ",\\n ".join(parts) + "\")."
        labels.append(label)

    return labels


In [5]:
import pandas as pd

def states_to_dataframe(states: list[dict[str, tuple[str, str]]]) -> pd.DataFrame:
    varnames = set()
    for state in states:
        varnames.update(state.keys())
    varnames = sorted(varnames)

    rows = []
    for i, state in enumerate(states):
        row = {"StateID": i}
        for varname in varnames:
            cell_value = state.get(varname, ("N/A", "N/A"))
            if len(cell_value) == 2:
                row[varname] = f"{cell_value[0]} ({cell_value[1]})"
            elif len(cell_value) == 1:
                row[varname] = cell_value[0]
        rows.append(row)

    return pd.DataFrame(rows)




In [6]:
# Reuse the above and find all the transitions between them
from typing import Any

def transitions_from_reference_model(
    RMODEL: list[str],
    ASPGARPFILES: list[str],
    state_descriptions: list[str],
) -> list[tuple[int, int]]:
    transitions: list[tuple[int, int]] = []

    ctl = clingo.Control(["0", "--project", "--warn=none"])

    for path in RMODEL + ASPGARPFILES:
        ctl.load(path)

    ctl.add("base", [], "\n".join(state_descriptions))
    ctl.add("base", [], ":- violation.")
    ctl.add("base", [], "#show state/2.")

    ctl.ground([("base", [])])

    with ctl.solve(yield_=True) as handle:
        for i, model in enumerate(handle):
            print(f"Model {i}:")
            print(model)

            from_state: int | None = None
            to_state: int | None = None

            for atom in model.symbols(shown=True):
                if atom.name != "state" or len(atom.arguments) != 2:
                    continue

                sid = atom.arguments[0]
                state_id = atom.arguments[1]

                if sid.type != clingo.SymbolType.Number:
                    continue
                if state_id.type != clingo.SymbolType.Number:
                    continue

                sid_int = sid.number
                state_int = state_id.number

                if sid_int == 0:
                    from_state = state_int
                elif sid_int == 1:
                    to_state = state_int

            if from_state is not None and to_state is not None:
                transitions.append((from_state, to_state))
            else:
                print("Warning: Could not find both from_state and to_state in the model.")

    return transitions

# Generate Full State Graph

In [7]:

states = states_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES)

df = states_to_dataframe(states)
display(df)

state_descriptions = states_to_description(states)
state_labels = states_to_labels(states)
with open(f"{OUTPUT}/states.lp", "w") as f:
    f.write("\n".join(state_descriptions))

transitions = transitions_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES, state_descriptions)

transition_descriptions = [f"edge(({from_state},{to_state}))." for from_state, to_state in transitions]

with open(f"{OUTPUT}/transitions.lp", "w") as f:
    f.write("\n".join(transition_descriptions))
    f.write("\n".join(state_labels))





,StateID,"(birth,death)",birth,death,population
0,0,eq,land(zero) (std),land(zero) (std),land(zero) (std)
1,1,eq,land(zero) (std),land(zero) (std),"interval(medium,inf) (std)"
2,2,gt,"interval(zero,inf) (std)",land(zero) (std),"interval(medium,inf) (std)"
3,3,eq,"interval(zero,inf) (std)","interval(zero,inf) (std)","interval(medium,inf) (std)"
4,4,gt,"interval(zero,inf) (dec)","interval(zero,inf) (dec)","interval(medium,inf) (dec)"
5,5,gt,"interval(zero,inf) (std)",land(zero) (std),"interval(zero,medium) (std)"
6,6,eq,land(zero) (std),land(zero) (std),"interval(zero,medium) (std)"
7,7,eq,"interval(zero,inf) (std)","interval(zero,inf) (std)","interval(zero,medium) (std)"
8,8,eq,"interval(zero,inf) (std)","interval(zero,inf) (std)",land(medium) (std)
9,9,gt,"interval(zero,inf) (std)",land(zero) (std),land(medium) (std)


UnboundLocalError: cannot access local variable 'parts' where it is not associated with a value

In [ ]:



def state_2_clingo_assumptions(state: dict[str, tuple[str, str]]):
    assumptions = []
    for varname, (value, direction) in state.items():
        assumptions.append((clingo.parse_term(f"holds({varname}, {value}, {direction})"), True))
    return assumptions


# Input: States as a Table

In [ ]:
df = states_to_dataframe(states)
display(df)

,StateID,"(birth,death)",birth,death,population
0,0,eq (None),land(zero) (std),land(zero) (std),land(zero) (std)
1,1,eq (None),land(zero) (std),land(zero) (std),"interval(medium,inf) (std)"
2,2,gt (None),"interval(zero,inf) (std)",land(zero) (std),"interval(medium,inf) (std)"
3,3,eq (None),"interval(zero,inf) (std)","interval(zero,inf) (std)","interval(medium,inf) (std)"
4,4,gt (None),"interval(zero,inf) (dec)","interval(zero,inf) (dec)","interval(medium,inf) (dec)"
5,5,gt (None),"interval(zero,inf) (std)",land(zero) (std),"interval(zero,medium) (std)"
6,6,eq (None),land(zero) (std),land(zero) (std),"interval(zero,medium) (std)"
7,7,eq (None),"interval(zero,inf) (std)","interval(zero,inf) (std)","interval(zero,medium) (std)"
8,8,eq (None),"interval(zero,inf) (std)","interval(zero,inf) (std)",land(medium) (std)
9,9,gt (None),"interval(zero,inf) (std)",land(zero) (std),land(medium) (std)


In [ ]:
def justify_state(state: dict[str, tuple[str, str]], reject = False, max_size=5):
    assumptions = state_2_clingo_assumptions(state)
    assumptions.append((clingo.parse_term("violation"), reject)) 
    for i in range(0,max_size):
        ctl = clingo.Control(["0", "--const", f"r={i}", "--project", "--warn=none"])
        for path in ASPGARP_FILES + META_FILES + BASIC_MODEL:
            ctl.load(path)
        ctl.add("base", [], "#show rule/3.")
        ctl.ground([("base", [])])
        success = False
        with ctl.solve(yield_=True,assumptions=assumptions) as handle:
            for model in handle:
                yield model.symbols(shown=True)
                success = True
            if success:
                return
    raise Exception(f"Could not justify state with r up to {max_size-1}.")

        

In [ ]:
# for i, model in enumerate(justify_state(states[2], reject=True)):
#     print(f"Model {i}:")
#     for atom in model:″
#         print(atom.arguments[2])
#     print("")

In [ ]:
# transition_descriptions = [f"edge(({from_state},{to_state}))." for from_state, to_state in transitions]

# #write the state and transition descriptions to files
# with open(f"{OUTPUT}/transitions.lp", "w") as f:
#     f.write("\n".join(transition_descriptions))
#     f.write("\n".join(state_labels))

# with open(f"{OUTPUT}/states.lp", "w") as f:
#     f.write("\n".join(state_descriptions))
    

!clingo --project --warn=none --outf=2 {OUTPUT}/transitions.lp  | clingraph --out=render --format=svg --viz-encoding="{project_root}/viz.lp" --dir={OUTPUT}/visuals --default-graph=states


-> Image for graph states, saved in: ./out/population/visuals/0/states.svg
